# IOAI — 2025 Regional Smart Cargo (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-smart-cargo.zip', '/tmp/sc.zip')
    with zipfile.ZipFile('/tmp/sc.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/train.csv' if os.path.exists('data/train.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Smart Cargo — 모범답안 (범주형 인코딩 + 선형회귀)
원대회 reference 접근: **`Weather` 더미 인코딩**을 추가해 선형회귀 성능을 끌어올립니다.
채점은 동일하게 held-out(`ID%5==0`) **MAE**(낮을수록 좋음).

In [ ]:
import pandas as pd, numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

df = pd.read_csv('data/train.csv')
is_val = df['ID'] % 5 == 0
tr, va = df[~is_val].reset_index(drop=True), df[is_val].reset_index(drop=True)

In [ ]:
def prep(d, cols):
    d = d.copy()
    dum = pd.get_dummies(d['Weather'], prefix='Weather', drop_first=True)
    X = pd.concat([d[['Distance','Time of Day','Traffic','Road Quality','Driver Experience']], dum], axis=1)
    return X.reindex(columns=cols, fill_value=0) if cols is not None else X

Xtr = prep(tr, None); cols = list(Xtr.columns)
Xva = prep(va, cols)
m = LinearRegression().fit(Xtr, tr['deliver_time'])
pred = m.predict(Xva)
print('held-out MAE:', round(mean_absolute_error(va['deliver_time'], pred), 4))
pd.DataFrame({'ID': va['ID'], 'deliver_time': pred}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(va))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)